## Tutorial 3: How important is the interaction between the hole and the electron?

Content:
- Example for SrTiO3.
    - Input preparation
    - Calculation for independent particles (IP)
    - Calculation for fully interacting two partcile problem (BSE)
    - Comparison with experiment

<img src="data/figures/img8.png" alt="drawing" width="1000"/>

In [3]:
import shore
from plotly import graph_objects as go
import numpy as np

-----

### Pipeline and server info

In [5]:
server=shore.RemoteServerManager(load='../test.server.pkl')
pipe=shore.Workflow(server=server)

----

#### Input prepartion

- structure

In [6]:
structure=shore.AtomicStructure('data/structures/sto.cif')

In [7]:
structure.view()

- photons

In [8]:
light=shore.Light()

In [9]:
light.add(shore.Photon(polarization=[0,0,1],
                        q=[1,0,0,],
                        energy=dict(edge='L2',element='Ti')))
light.add(shore.Photon(polarization=[0,1,0],
                        q=[0,0,1,],
                        energy=dict(edge='L2',element='Ti')))
light.add(shore.Photon(polarization=[1,0,0],
                        q=[0,0,1,],
                        energy=dict(edge='L2',element='Ti')))

In [10]:
light.show()

Light Bundle Information:
1: Photon(polarization=[0, 0, 1], q=[1, 0, 0], energy=460.2 eV, light-matter interaction=dipole)
2: Photon(polarization=[0, 1, 0], q=[0, 0, 1], energy=460.2 eV, light-matter interaction=dipole)
3: Photon(polarization=[1, 0, 0], q=[0, 0, 1], energy=460.2 eV, light-matter interaction=dipole)


- info about material:

In [11]:
matter=shore.Matter(structure=structure,load='data/inputs/sto.param',
                    )

In [12]:
matter.show()

<IPython.core.display.JSON object>

- Input object:

In [13]:
basic=shore.Input(name='basic',matter=matter, light=light)

------

### Calculation round 1

Do fork for new input with no interaction specified `bse_core_strength=0`

In [14]:
input_ip=basic.fork(name='ip', bse_core_strength=0,)

Create calculation instance

In [15]:
ipcalc=shore.Calculation(workflow=pipe,input=input_ip)

Run calculations

In [16]:
ipcalc.run(overwrite=True)

Job submitted successfully with Job ID: 28096


 Check status

In [17]:
ipcalc.get_status()

parsing:   0%|          | 0/3 [00:00<?, ?it/s]

atomic:   0%|          | 0/2 [00:00<?, ?it/s]

dft:   0%|          | 0/2 [00:00<?, ?it/s]

prep:   0%|          | 0/2 [00:00<?, ?it/s]

screen:   0%|          | 0/2 [00:00<?, ?it/s]

bse:   0%|          | 0/2 [00:00<?, ?it/s]

Errors:
Loading q-ch/qe/7.3.1/gcc/11.2/mpich/mkl
  Loading requirement: devtools/math/mkl/2024.1.0.695
    devtools/mpi/mpich/4.2.1/gcc/11.2
DFT Stage Failed

Messages:

Now is runing:
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
 


Retrieve data from the cluster

In [18]:
# ipres=ipcalc.sync()
ipres=shore.ResultsHandler().load('./data/ref/sto/ip.pkl')

Checking workflow

In [ ]:
pipe.show()

Save results and check out local folder SrTiO3 

In [ ]:
# ipres.save()

Plot results

In [19]:
fig=go.Figure()
ipres.plot_xas(fig=fig, lw=4,)
fig.update_layout(xaxis=dict(range=[-10, 30])) 

----

### Full BSE calculations

Create new input

In [20]:
bse=input_ip.fork(name='bse', bse_core_strength=1)

Checkout that everything is in place

In [21]:
bse.info()

Atomic Structure Information:
Attributes:
  - content: <shore.input_manager.ocean_input object at 0x168dc4e30>
  - light: <shore.input_manager.Light object at 0x168dc4e60>
  - matter: <shore.input_manager.Matter object at 0x168c37ce0>
  - name: bse
  - structure: <shore.AtomicStructure object at 0x168c36cc0>
  - target: xas
Methods:
  - fork
  - info



Create new calculation instance

In [22]:
bsecalc=shore.Calculation(workflow=pipe, input=bse)

Sumbit the calculations to the cluster

In [ ]:
bsecalc.run()

Get status and wait untill it's done

In [ ]:
bsecalc.get_status()

Results

In [ ]:
bseres=bsecalc.sync()

Save results

In [ ]:
bseres.save()

Checkout pipeline

In [ ]:
pipe.show()

In case it takes longer then it should load results from `data/ref` folder

In [23]:
bseres=shore.ResultsHandler(load='data/ref/sto/bse.pkl')

In [24]:
fig=go.Figure()
ipres.plot_xas(fig=fig, element='Ti', core_level='2p', site_number=1, name='IP', lw=5)
bseres.plot_xas(fig=fig, element='Ti', core_level='2p', site_number=1, name='BSE', lw=5)
fig.show()

----

### Comparison to the experiment

In [25]:
exp=np.loadtxt('./data/ref/sto/exp_sto')

In [26]:
fig=go.Figure()
bseres.plot_xas(fig=fig, element='Ti', core_level='2p', site_number=1, name='BSE', lw=5)
fig.add_trace(go.Scatter(x=exp[0], y=exp[1]/max(exp[1]), mode='markers',  marker=dict(size=10,color='grey',opacity=0.8,line=dict(width=1, color='black')),name='Experimental Data [1]' ))
fig.update_layout(xaxis=dict(range=[-10, 10])) 

[1] Bhogra et al. Scientific Reports. 2019